[![在 Colab 中打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Arize-ai/phoenix/blob/main/tutorials/agents/openai/openai_agents_orchestrator.ipynb)


# <center>OpenAI 智能体模式：编排器与工作智能体</center>

本入门指南介绍如何使用 `openai-agents` 库构建智能体循环。

这一模式使用编排器和工作智能体。编排器为特定子任务选择工作智能体；工作智能体尝试完成任务并返回结果。编排器再根据结果选择下一个工作智能体，直到得到最终结果。

下面构建一个根据用户投资策略创建股票和 ETF 投资组合的智能体。
1. **编排器**：根据用户投资策略选择工作智能体。
2. **研究智能体**：在网上搜索能够支持用户投资策略的股票和 ETF 信息。
3. **评估智能体**：评估研究报告，并反馈缺失的数据。
4. **投资组合智能体**：根据研究报告创建股票和 ETF 投资组合。

### 安装库

In [ ]:
# 安装 OpenAI 基础库
!pip install -q openai openai-agents

# 安装 OpenInference/OpenTelemetry 追踪的可选库
!pip install -q arize-phoenix-otel openinference-instrumentation-openai-agents openinference-instrumentation-openai

### 设置密钥
将 OpenAI API 密钥添加到环境变量 `OPENAI_API_KEY`。

从 [app.phoenix.arize.com](https://app.phoenix.arize.com) 的设置页面复制 Phoenix `API_KEY`。

In [ ]:
import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("🔑 请输入 OpenAI API 密钥： ")

if "PHOENIX_API_KEY" not in os.environ:
    os.environ["PHOENIX_API_KEY"] = getpass("🔑 请输入 Phoenix API 密钥： ")

if "PHOENIX_COLLECTOR_ENDPOINT" not in os.environ:
    os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = getpass("🔑 请输入 Phoenix 收集器端点： ")

### 设置追踪

In [ ]:
from phoenix.otel import register

tracer_provider = register(
    project_name="openai-agents",
    auto_instrument=True,
)

## 创建智能体

In [ ]:
from pprint import pprint
from textwrap import dedent
from typing import Literal

from agents import Agent, Runner, TResponseInputItem, WebSearchTool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field


class PortfolioItem(BaseModel):
    ticker: str = Field(description="股票或 ETF 的代码。")
    allocation: float = Field(
        description="该代码在投资组合中的分配百分比。所有分配比例总和应为 100。"
    )
    reason: str = Field(description="将该代码纳入投资组合的原因。")


class Portfolio(BaseModel):
    tickers: list[PortfolioItem] = Field(
        description="能够支持用户声明的投资策略的股票代码列表。"
    )


class EvaluationFeedback(BaseModel):
    feedback: str = Field(
        description="根据用户投资策略创建股票和 ETF 投资组合还缺少哪些数据。"
    )
    score: Literal["pass", "needs_improvement", "fail"] = Field(
        description="研究报告评级。至少有 5 个代码，且数据支持按用户投资策略创建投资组合时为 pass；支持数据不足时为 needs_improvement；没有代码时为 fail。"
    )


evaluation_agent = Agent(
    name="Evaluation Agent",
    instructions=dedent(
        """你是资深金融分析师。你会收到包含正面与负面催化因素的股票研究报告。请评估报告并反馈需要改进之处。"""
    ),
    model="gpt-4.1",
    output_type=EvaluationFeedback,
)

portfolio_agent = Agent(
    name="Portfolio Agent",
    instructions=dedent(
        """你是资深金融分析师。你会收到股票研究报告。请创建能够支持用户声明的投资策略的股票和 ETF 投资组合，并在分配理由中引用研究报告的事实和数据。"""
    ),
    model="o4-mini",
    output_type=Portfolio,
)

research_agent = Agent(
    name="FinancialSearchAgent",
    instructions=dedent(
        """你是专注金融主题的研究助手。给定股票代码后，使用网络搜索获取最新背景，生成不超过 50 个词的简短摘要，重点关注对金融分析师有用的关键数字、事件或引述。"""
    ),
    model="gpt-4.1",
    tools=[WebSearchTool()],
    model_settings=ModelSettings(tool_choice="required", parallel_tool_calls=True),
)

orchestrator_agent = Agent(
    name="Routing Agent",
    instructions=dedent("""你是资深金融分析师，正在根据我声明的投资策略创建投资组合。你的任务是转交给适当的智能体或工具。

    首先转交 research_agent，获取能够支持用户投资策略的股票和 ETF 报告。
    然后转交 evaluation_agent，对报告评分。如果返回 needs_improvement 或 fail，继续使用 research_agent 收集更多信息。
    当 evaluation_agent 返回 pass 时，转交 portfolio_agent 创建投资组合。"""),
    model="gpt-4.1",
    handoffs=[
        research_agent,
        evaluation_agent,
        portfolio_agent,
    ],
)

In [ ]:
user_input = input("请输入投资策略： ")
input_items: list[TResponseInputItem] = [{"content": user_input, "role": "user"}]

while True:
    orchestrator = await Runner.run(orchestrator_agent, input_items)
    orchestrator_output = orchestrator.final_output
    pprint(orchestrator_output)

    input_items = orchestrator.to_input_list()
    if isinstance(orchestrator_output, Portfolio):
        break
    print("返回编排器")
    # input_items.append({"content": f"继续", "role": "user"})

print("智能体执行完成")